# plotmux — case study: reproducing bokeh's slope example

This notebook reproduces [bokeh's `slope` annotation example](https://docs.bokeh.org/en/latest/docs/examples/basic/annotations/slope.html) -- scatter markers with a separate yellow fill / black edge drawn with `alpha=0.8`, plus a dashed reference line of gradient 2 and y-intercept 10 at `line_width=4`, on a figure with a light-gray background and `y_range.start = 0` -- through plotmux's unified API, unchanged, on all five backends.

A slope has no data of its own, so it is drawn over the x extent of its panel: the `xmin`/`xmax` of the panel where they are set, else the extent of the data of the other marks. The scatter + slope combination below has a data-bound sibling, so the *same* `plotmux.layer(...)` call reproduces the example unchanged on every backend; a standalone `plotmux.slope(...)` needs `xmin` and `xmax`, and works on every backend too (see the end of the notebook).

In [ ]:
import numpy as np

import plotmux
from plotmux.figure import Figure
from plotmux.specs import ScatterSpec, SlopeSpec

# Bokeh renders lazily: its native figure's `_repr_html_` only shows a plain
# text description of the model until `output_notebook()` registers Bokeh's
# own IPython display hook (BokehJS), which is what actually draws the plot
# inline. This is harmless to call even for notebook runs that never pick
# the bokeh backend.
try:
    from bokeh.io import output_notebook

    output_notebook()
except ImportError:
    pass

## Reproducing bokeh's own generated data

Same equation (`gradient=2, intercept=10`), same `x` range, and the same additive Gaussian noise bokeh's own example uses to scatter the points around the line.

In [ ]:
rng = np.random.default_rng(42)

gradient, intercept = 2, 10
x = np.arange(0, 20, 0.2)
y = gradient * x + intercept + rng.normal(0, 4, size=x.shape)

## Same call, every backend

The cells below rerun the *exact same* `plotmux.layer(...)` call across all five backends to confirm the reproduction is unchanged, not backend-specific code paths in disguise.

In [ ]:
def make_slope_figure(x: np.ndarray, y: np.ndarray, gradient: float, intercept: float) -> Figure:
    r"""Build the scatter + slope figure reproducing bokeh's example."""
    return plotmux.layer(
        ScatterSpec(
            x=x,
            y=y,
            color="yellow",
            edgecolor="black",
            size=8,
            alpha=0.8,
        ),
        SlopeSpec(
            gradient=gradient,
            intercept=intercept,
            color="#5386a3",
            linestyle="dashed",
            linewidth=4,
        ),
        background_color="#fafafa",
        ymin=0,
    )

### matplotlib

In [ ]:
with plotmux.backend("matplotlib"):
    fig_mpl = make_slope_figure(x, y, gradient, intercept)
fig_mpl.to_native()

### bokeh

In [ ]:
from bokeh.io import show

with plotmux.backend("bokeh"):
    fig_bokeh = make_slope_figure(x, y, gradient, intercept)
show(fig_bokeh.to_native())

### altair

The slope is drawn as the segment over the x extent of the scatter data.

In [ ]:
with plotmux.backend("altair"):
    fig_altair = make_slope_figure(x, y, gradient, intercept)
fig_altair.to_native()

### xy

Same as altair: the slope is the segment over the x extent of the scatter data.

In [ ]:
with plotmux.backend("xy"):
    fig_xy = make_slope_figure(x, y, gradient, intercept)
fig_xy.to_native()

### plotly

In [ ]:
with plotmux.backend("plotly"):
    fig_plotly = make_slope_figure(x, y, gradient, intercept)
fig_plotly.to_native()

## A standalone slope needs `xmin` and `xmax`

A slope with no data-bound sibling has no x extent. It raises an `InvalidSpecError` when the panel is built, the same error on every backend:

In [ ]:
from plotmux.exceptions import InvalidSpecError

try:
    plotmux.slope(gradient, intercept)
except InvalidSpecError as error:
    print(error)

With `xmin` and `xmax`, the standalone slope is the same segment on every backend (`matplotlib` and `bokeh` draw a native infinite line that keeps extending when you zoom):

In [ ]:
for backend_name in ["matplotlib", "bokeh", "altair", "xy", "plotly"]:
    with plotmux.backend(backend_name):
        fig_standalone = plotmux.slope(
            gradient, intercept, xmin=0, xmax=20, linestyle="dashed", linewidth=4
        )
    print(backend_name, type(fig_standalone.to_native()).__name__)